# PII Guardrail Model Training

Fine-tune Llama 3.2-1B for PII detection using QLoRA with Unsloth.

**Target Hardware**: Google Colab T4 GPU (16GB VRAM)

**Model**: `unsloth/Llama-3.2-1B-Instruct`

**Technique**: QLoRA (4-bit quantization with LoRA adapters)

## Features
- Detects Indian PII: Aadhaar, PAN
- Detects General PII: Email, Phone, Names, SSN, Credit Cards
- Outputs JSON with entity types, positions, confidence, and reasons


In [ ]:
# Cell 1: Install Unsloth and dependencies
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers scikit-learn


In [ ]:
# Cell 2: Mount Google Drive and setup configuration
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import torch
from datetime import datetime

# Paths - Your uploaded data folder in Google Drive
DATA_DIR = "/content/drive/MyDrive/pii-guardrail-model"
OUTPUT_DIR = "/content/drive/MyDrive/pii-guardrail-model/output"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configuration
CONFIG = {
    "model_name": "unsloth/Llama-3.2-1B-Instruct",
    "max_seq_length": 512,
    "load_in_4bit": True,
    "lora_rank": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "num_epochs": 3,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.1,
    "data_dir": DATA_DIR,
    "output_dir": OUTPUT_DIR,
}

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify data directory exists and list contents
print(f"\nData directory: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    print("Contents:")
    for f in os.listdir(DATA_DIR):
        filepath = os.path.join(DATA_DIR, f)
        size = os.path.getsize(filepath) if os.path.isfile(filepath) else "DIR"
        print(f"  {f}: {size if isinstance(size, str) else f'{size/1024:.1f} KB'}")
else:
    print("WARNING: Data directory not found! Please check the path.")


In [ ]:
# Cell 3: Load Model with QLoRA (4-bit quantization)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,  # Auto-detect
)

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Model parameters: {model.num_parameters():,}")


In [ ]:
# Cell 4: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")


In [ ]:
# Cell 5: Load Training Data from Google Drive
# Data was pre-generated and uploaded to: My Drive/pii-guardrail-model/

import os

# Define paths to your uploaded data files
TRAIN_FILE = os.path.join(CONFIG["data_dir"], "train.jsonl")
EVAL_FILE = os.path.join(CONFIG["data_dir"], "eval.jsonl")

# Check if files exist
print("Checking for training data files...")
print(f"  Train file: {TRAIN_FILE}")
print(f"  Eval file: {EVAL_FILE}")

if not os.path.exists(TRAIN_FILE):
    raise FileNotFoundError(f"Training file not found: {TRAIN_FILE}\nPlease ensure train.jsonl is in your Google Drive folder.")
if not os.path.exists(EVAL_FILE):
    raise FileNotFoundError(f"Evaluation file not found: {EVAL_FILE}\nPlease ensure eval.jsonl is in your Google Drive folder.")

# Load and count samples
train_data = []
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            train_data.append(json.loads(line))

eval_data = []
with open(EVAL_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            eval_data.append(json.loads(line))

print(f"\n✓ Loaded {len(train_data)} training samples")
print(f"✓ Loaded {len(eval_data)} evaluation samples")

# Analyze distribution
difficulty_counts = {'positive': 0, 'negative': 0, 'dangerous_neg': 0}
for item in train_data:
    try:
        output = json.loads(item['conversations'][2]['content'])
        if output.get('flagged', False):
            difficulty_counts['positive'] += 1
        elif 'not sensitive' in output.get('reason', ''):
            difficulty_counts['dangerous_neg'] += 1
        else:
            difficulty_counts['negative'] += 1
    except:
        difficulty_counts['positive'] += 1  # Default

print(f"\nTraining Data Distribution:")
print(f"  Positive (contains PII): {difficulty_counts['positive']}")
print(f"  Negative (no PII): {difficulty_counts['negative']}")
print(f"  Dangerous Negatives (PII-like but not PII): {difficulty_counts['dangerous_neg']}")

# Preview a sample
print("\n--- Sample Preview ---")
print(json.dumps(train_data[0], indent=2)[:500] + "...")

# Define system prompt for evaluation/inference later
SYSTEM_PROMPT = """You are a PII detection model. Analyze text and identify PII entities with their exact positions.

Output JSON with: flagged (bool), entities (array with type, value, start, end), confidence (0-1), reason (string).

Entity types: IN_AADHAAR, IN_PAN, EMAIL_ADDRESS, PHONE_NUMBER, PERSON, US_SSN, CREDIT_CARD

Important:
- Report confidence based on how clear/corrupted the PII appears
- Handle typos, OCR errors, and obfuscation attempts
- Don't flag non-PII identifiers like order IDs, product codes, or dates"""


In [ ]:
# Cell 6: Load and format datasets from Google Drive
from datasets import load_dataset

# Load from Google Drive paths
train_dataset = load_dataset('json', data_files=TRAIN_FILE, split='train')
eval_dataset = load_dataset('json', data_files=EVAL_FILE, split='train')

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Eval dataset: {len(eval_dataset)} samples")

def format_chat(example):
    """Format conversations into the Llama chat template."""
    formatted = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

train_dataset = train_dataset.map(format_chat)
eval_dataset = eval_dataset.map(format_chat)

print("\nFormatted sample preview:")
print(train_dataset[0]['text'][:500])


In [ ]:
# Cell 7: Setup Training
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    args=training_args,
)

print("Trainer initialized!")
print(f"Training for {CONFIG['num_epochs']} epochs")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")


In [ ]:
# Cell 8: Train the model
print(f"Starting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

trainer_stats = trainer.train()

print(f"\nTraining completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Training loss: {trainer_stats.training_loss:.4f}")


In [ ]:
# Cell 9: Evaluation - Calculate Precision, Recall, F1 per entity type
from sklearn.metrics import precision_recall_fscore_support
from collections import defaultdict

def extract_entities_from_output(output_text):
    """Extract entities from model output JSON."""
    try:
        start = output_text.find('{')
        end = output_text.rfind('}') + 1
        if start >= 0 and end > start:
            data = json.loads(output_text[start:end])
            return data.get('entities', []), data.get('flagged', False)
    except:
        pass
    return [], False

def evaluate_model(model, tokenizer, eval_samples, max_samples=100):
    """Evaluate model on eval set."""
    FastLanguageModel.for_inference(model)
    
    results = {
        'true_flagged': [],
        'pred_flagged': [],
        'true_entities': [],
        'pred_entities': [],
    }
    
    for i, sample in enumerate(eval_samples[:max_samples]):
        if i % 20 == 0:
            print(f"Evaluating sample {i+1}/{max_samples}...")
        
        conversations = sample['conversations']
        ground_truth = json.loads(conversations[2]['content'])
        
        messages = conversations[:2]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        pred_entities, pred_flagged = extract_entities_from_output(output_text)
        
        results['true_flagged'].append(ground_truth['flagged'])
        results['pred_flagged'].append(pred_flagged)
        results['true_entities'].append(ground_truth['entities'])
        results['pred_entities'].append(pred_entities)
    
    return results

print("Running evaluation...")
eval_results = evaluate_model(model, tokenizer, eval_data, max_samples=100)


In [ ]:
# Cell 10: Print Precision-Recall Report
precision, recall, f1, _ = precision_recall_fscore_support(
    eval_results['true_flagged'],
    eval_results['pred_flagged'],
    average='binary'
)

print("=" * 60)
print("PII DETECTION METRICS - PRECISION/RECALL REPORT")
print("=" * 60)
print(f"\nOverall Flagged Detection:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# Per-entity-type metrics
entity_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

for true_ents, pred_ents in zip(eval_results['true_entities'], eval_results['pred_entities']):
    true_set = {(e['type'], e['value']) for e in true_ents}
    pred_set = {(e['type'], e['value']) for e in pred_ents}
    
    for etype, value in true_set:
        if (etype, value) in pred_set:
            entity_metrics[etype]['tp'] += 1
        else:
            entity_metrics[etype]['fn'] += 1
    
    for etype, value in pred_set:
        if (etype, value) not in true_set:
            entity_metrics[etype]['fp'] += 1

print(f"\nPer-Entity-Type Metrics:")
print("-" * 60)
print(f"{'Entity Type':<20} {'Precision':>12} {'Recall':>12} {'F1':>12}")
print("-" * 60)

for etype in sorted(entity_metrics.keys()):
    counts = entity_metrics[etype]
    tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"{etype:<20} {prec:>12.4f} {rec:>12.4f} {f1_score:>12.4f}")

print("=" * 60)


In [ ]:
# Cell 11: Save LoRA adapters
lora_path = f"{CONFIG['output_dir']}/lora-adapters"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

print(f"LoRA adapters saved to: {lora_path}")

# Save training config and metrics
metrics = {
    "config": CONFIG,
    "training_loss": trainer_stats.training_loss,
    "evaluation": {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    },
    "entity_metrics": {k: dict(v) for k, v in entity_metrics.items()},
    "timestamp": datetime.now().isoformat(),
}

metrics_path = f"{CONFIG['output_dir']}/training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved to: {metrics_path}")


In [ ]:
# Cell 12: Test Inference with sample inputs
FastLanguageModel.for_inference(model)

test_texts = [
    "My Aadhaar number is 2345 6789 0123.",
    "Contact me at rahul.sharma@gmail.com or call +91 98765 43210.",
    "My PAN card number is ABCPD1234E for tax filing.",
    "The weather today is sunny with a high of 25 degrees.",
    "User Darshan Krishna with Aadhaar 9876 5432 1098 and PAN BNZPM2501F registered.",
]

print("=" * 70)
print("TEST INFERENCE RESULTS")
print("=" * 70)

for text in test_texts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Detect PII in: "{text}"'}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"\nInput: {text}")
    print(f"Output:\n{output_text}")
    print("-" * 70)


In [ ]:
# Cell 13: Export for deployment - zip LoRA adapters
import shutil

zip_path = "/content/pii-guardrail-lora"
shutil.make_archive(zip_path, 'zip', lora_path)

print(f"Created: {zip_path}.zip")
print("\nTo download, uncomment and run:")
print("# from google.colab import files")
print(f"# files.download('{zip_path}.zip')")
